# CareSignal: evidence and correction walkthrough

This notebook calls the same package functions as the Streamlit screen. It uses fictional residents and notes for the workflow, then runs the phrase matcher on frozen development inputs. A successful Gemini development check is recorded separately in `docs/gemini-development-results/`; public held-out performance and OpenAI live inference remain pending.

This project is a prototype and is not medical advice.


In [ ]:
import sys
import tempfile
from pathlib import Path

ROOT = Path.cwd()
if not (ROOT / "caresignal").exists():
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

In [ ]:
from caresignal.evaluation import evaluate_examples, load_dataset, summary_markdown, write_results
from caresignal.extractors import FixtureExtractor, PhraseMatcher
from caresignal.fixtures import DEMO_REPLACEMENT
from caresignal.service import CareSignalService

## One note, two confirmed baselines

The note documents an aide beside the resident on the breakfast walk. Elena's confirmed usual level is Independent; Ruth's is Staff beside or supervising. The model does not compare the levels: code does.


In [ ]:
demo_directory = tempfile.TemporaryDirectory(prefix="caresignal-notebook-")
service = CareSignalService(Path(demo_directory.name) / "demo.sqlite3", "notebook-fictional-demo")
service.reset_demo()
residents = {resident.display_name: resident for resident in service.list_residents()}
cases = {name: service.list_cases(resident.id)[0] for name, resident in residents.items()}
for case in cases.values():
    service.analyze_active_revision(case.id, FixtureExtractor())
views = {name: service.get_handoff_view(case.id) for name, case in cases.items()}
assert views["Elena"].revision.body == views["Ruth"].revision.body
assert views["Elena"].comparison.status == "MORE_HELP"
assert views["Ruth"].comparison.status == "MATCHES"
[
    (name, view.run.result.label.value, view.comparison.status, view.baseline.level.value)
    for name, view in views.items()
]

## Replace the source, preserve the original

The replacement becomes active before inference starts. A walker alone leaves human assistance unspecified. The old flag and excerpt cannot remain on the current card.


In [ ]:
original = views["Elena"].revision
replacement = service.replace_active_revision(cases["Elena"].id, original.id, DEMO_REPLACEMENT)
pending_view = service.get_handoff_view(cases["Elena"].id)
assert pending_view.revision.id == replacement.id
assert pending_view.run is None
service.analyze_active_revision(cases["Elena"].id, FixtureExtractor())
corrected_view = service.get_handoff_view(cases["Elena"].id)
assert corrected_view.comparison.status == "UNCLEAR"
assert corrected_view.run.revision_id == replacement.id
[
    (
        revision.version,
        "Active" if revision.id == corrected_view.revision.id else "Corrected",
        revision.body,
    )
    for revision in service.history(cases["Elena"].id)
]

## Frozen fictional development checks

These scores measure the phrase matcher's behavior on authored development examples, including deliberate distractors. They are not a held-out public experiment. Exact evidence requires the entire frozen span set, so a correct label with a different excerpt can fail the joint score.


In [ ]:
examples = load_dataset(ROOT / "data/fictional_development.jsonl")
report = evaluate_examples(examples, PhraseMatcher())
print(summary_markdown(report))
artifacts = write_results(report, ROOT / "artifacts/notebook-fictional-development")
artifacts

## Public synthetic development example: Fruta `nn_1093`

The original Dutch text and source dataset revision are frozen in `data/fruta_nn_1093.json`. The English text uses coding-agent-authored translation `caresignal-agent-en-literal-v1`, which has not been independently reviewed. This disclosed development example expects Hands-on help with evidence spanning both sentences. It is excluded from any held-out claim.


In [ ]:
public_examples = load_dataset(ROOT / "data/fruta_development.jsonl")
public_report = evaluate_examples(public_examples, PhraseMatcher())
print(summary_markdown(public_report))
public_report["examples"][0]

## Recorded Gemini development check; held-out experiment pending

The actual Gemini `gemini-3.8-flash` run on `nn_1093` returned Hands-on help with exact full two-sentence evidence (1/1 joint correct; errors 0). See `docs/gemini-development-results/results.json`. It used `gemini-structured-v1`, shared prompt `walking-assistance-v1`, and the coding-agent-authored English translation, which has not been independently reviewed. This single disclosed development check does not estimate held-out performance. OpenAI live inference remains pending.

To rerun a real structured-output adapter, configure `GEMINI_API_KEY` (optional `GEMINI_MODEL`) or `OPENAI_API_KEY` (optional `OPENAI_MODEL`), then select `--provider gemini` or `--provider openai` with the CLI's `--mode live` or `--mode compare`. The CLI defaults to OpenAI when the provider option is omitted. Do not execute live code merely by opening this notebook.

```bash
python -m caresignal.evaluation --dataset data/fruta_development.jsonl --mode live --provider gemini --output-dir artifacts/fruta-development-gemini
python -m caresignal.evaluation --dataset data/fruta_held_out.jsonl --mode compare --provider gemini --output-dir artifacts/fruta-held-out-gemini
```

The second command requires a separately frozen, translated, annotated held-out file that is not yet supplied. Inspect and report at least one real held-out error after that run; no example or score is invented here.
